# Train một lượt lưới RL trên Colab (bản dựng từ `_bundles/colab_luoi/make_colab_nb.py`, dữ liệu của `uyenle0712`)

Dành cho lượt dài mà Kaggle phải nối nhiều commit, ví dụ PPO 1.000 bước (khoảng 41 h trên T4 [suy, probe 3/11]), GSPO 1.000 bước, DPO.
Cùng script, cùng tên thư mục ra như notebook Kaggle (`make_nb.py`), nên điểm lưu hai bên dùng chung được.

**Cách chạy**
1. Sửa ô 1: `LUOT` (cú pháp `algo:reward:seed[:k=v,...,tag=...]`) và `CHE_DO`.
2. Lần đầu với một lượt: `CHE_DO = "tham_do"` trên **L4** → Run all (khoảng 30–60 phút). Ô 5 in s/bước, đỉnh VRAM trên 50 câu nhắc dài nhất và khuyến nghị máy.
   Muốn so với A100 thì đổi máy, chạy thăm dò lại; kết quả hai lần lưu trên Drive và ô 5 tự tính tỉ số. Luật: tỉ số giờ L4/A100 ≤ 1,5 thì dùng L4.
3. Đổi `CHE_DO = "train"` → Run all. Ô 6 chạy foreground suốt lượt: giữ tab mở, đừng bấm Stop. Stop cũng không giết tiến trình train (`start_new_session=True`),
   nhưng ô theo dõi mất thì Colab ngắt máy sau khoảng 90 phút không tương tác.
4. **Mất máy:** bật lại đúng máy cũ, Run all. Ô 6 chép điểm lưu mới nhất từ Drive về rồi chạy tiếp, và kiểm dòng `[nạp default] … → x>0`.
   Nếu tiến trình cũ còn sống (nhân Python restart nhưng máy chưa mất), ô 6 chỉ theo dõi tiếp, không chạy bản thứ hai.
5. Xong: ô 7 zip kết quả lên `MyDrive/thesis/luoi_colab/<lượt>/`. Tải zip đó về, đặt vào `runs/luoi/<đợt>/`.

Điểm lưu: mỗi 25 bước (mặc định của script); đồng bộ lên Drive mỗi 5 phút bằng chép sang tên tạm rồi đổi tên; Drive giữ 3 điểm lưu cuối.
Khoá Kaggle chỉ ghi vào `/root/.kaggle`, không in, không lưu vào Drive hay notebook. Đặt sẵn ở Colab Secrets tên `KAGGLE_TOKEN` cho khỏi gõ.


In [ ]:
# Ô 1. Cấu hình — chỉ sửa ô này
LUOT = "ppo:spice:101:mu=2,lam=1.0,steps=1000,tag=l1"   # ví dụ khác: "gspo:spice:101:steps=1000,tag=s1000" · "dpo_on:spice:101" · "grpo:spice_cong:101" · "seqgan:spice:101" · "grpo:bcmr:101" · "es:spice:101:es_sigma=0.002,tag=s2"
CHE_DO = "tham_do"            # "tham_do" (20 bước trên 50 câu nhắc dài nhất, không lưu) hoặc "train"
CAT_GIO = None                # trần giờ của phiên (None = chạy tới hết); chạm trần thì dừng nhẹ và điểm lưu cuối đã lên Drive
TK = "uyenle0712"              # tài khoản Kaggle chứa dữ liệu (fgrb-p1-bundle, grpo-luoi-script, luoi-kit)
DRIVE_GOC = "/content/drive/MyDrive/thesis/luoi_colab"


In [ ]:
# Ô 2. Logic (nhúng từ colab_logic.py) + kiểm cấu hình, chưa cài gì
import os, sys, time
T0 = time.time()
W = '/content/w'; os.makedirs(W, exist_ok=True)
open(f'{W}/colab_logic.py', 'w', encoding='utf-8').write('''# -*- coding: utf-8 -*-
"""Phần logic của notebook Colab train một lượt lưới RL (colab_luoi_train.ipynb). Tách ra để kiểm được trên
Kaggle CPU bằng tiến trình giả (kiem/kiem_logic.py), không cần GPU hay Colab.

Hàm chính:
  phan_tich(luot)            "algo:reward:seed[:k=v,...]" → (algo, reward, seed, opt)
  tep_cua(luot)              script phải gọi (grpo_luoi / grpo_codien / grpo_ppo / grpo_moi_b1 / thuong_moi / sft_tiep)
  ten_thu_muc(luot, probe)   tên thư mục ra, trùng quy ước make_nb.py (để điểm lưu Kaggle ↔ Colab dùng chung được)
  dung_lenh(...)             dòng lệnh đầy đủ
  diem_luu_du(d)             điểm lưu đủ ba tệp, xếp theo bước
  dong_bo_len(out, dr)       chép điểm lưu mới lên Drive: copytree sang tên tạm rồi đổi tên, giữ K bản cuối
  khoi_phuc(dr, out)         máy mới: chép điểm lưu đủ mới nhất từ Drive về
  tien_do(log)               đọc train.log: số bước, s/bước, [tiếp từ], [nạp default], lỗi
  chon_may(...)              luật chọn L4 hay A100 (CLAUDE.md: tỉ số giờ ≤ 1,5 thì L4; đỉnh VRAM > 17 GiB thì không L4)
  chay(cmd, log, ...)        Popen start_new_session=True, stdout ghi ra tệp local
  theo_doi(...)              vòng foreground: in mỗi in_moi giây, đồng bộ Drive mỗi dong_bo_moi giây, dừng ở trần giờ
"""
import os, re, ast, glob, json, time, shutil, signal, subprocess, statistics as st

CU = ("grpo", "drgrpo", "rloo", "reinforce", "scst")                       # grpo_luoi.py
CD = ("vanilla", "mad", "rft", "cem", "awr", "raml", "rank")                 # grpo_codien.py
B1 = ("dpo_on", "dpo_off", "rft_off", "mrt", "spg", "drgrpo_full", "grpo_seq", "gspo")   # grpo_moi_b1.py
PP = ("ppo", "a2c")                                                          # grpo_ppo.py
B2 = ("rlce", "mixer", "seqgan", "quark", "es")                              # grpo_moi_b2.py (cùng thưởng bcmr)
THUONG_CU = ("spice", "cider", "bleu4", "rougel", "chrf", "meteor", "bertscore", "spider")
THUONG_B2 = ("bcmr",)                                                        # grpo_moi_b2.py, chỉ với --algo grpo
THUONG_MOI = ("spice_cong", "ten_loai", "loai")                              # thuong_moi.py
CO = {"beta": "--beta", "loss": "--loss-type", "scale": "--scale-rewards", "mu": "--mu", "lam": "--lam",
      "is": "--is-level", "eps": "--eps-high", "steps": "--max-steps", "G": "--G", "lr": "--lr",
      "dpo_beta": "--dpo-beta", "mrt_alpha": "--mrt-alpha", "spg_tau": "--spg-tau", "eps_low": "--eps-low",
      "ce_lam": "--ce-lam", "mixer_rho0": "--mixer-rho0", "mixer_frac": "--mixer-frac", "gan_mix": "--gan-mix",
      "gan_warm": "--gan-warm", "gan_lr": "--gan-lr", "quark_k": "--quark-k", "quark_pool": "--quark-pool",
      "es_pop": "--es-pop", "es_m": "--es-m", "es_sigma": "--es-sigma", "es_lr": "--es-lr", "es_save": "--es-save"}
CO_GL = {"beta", "steps", "G", "lr"}           # cờ mà grpo_luoi.py (và thuong_moi.py) nhận
CO_B1 = {"dpo_beta", "mrt_alpha", "spg_tau", "eps_low"}
CO_B2 = {"ce_lam": "rlce", "mixer_rho0": "mixer", "mixer_frac": "mixer", "gan_mix": "seqgan", "gan_warm": "seqgan",
         "gan_lr": "seqgan", "quark_k": "quark", "quark_pool": "quark", "es_pop": "es", "es_m": "es", "es_sigma": "es",
         "es_lr": "es", "es_save": "es"}
TEP_CAN = ("trainer_state.json", "optimizer.pt", "adapter_model.safetensors")


def phan_tich(luot):
    al, rw, sd, *x = luot.split(":")
    opt = dict(kv.split("=", 1) for kv in x[0].split(",")) if x and x[0] else {}
    assert set(opt) <= set(CO) | {"tag"}, f"cờ lạ trong {luot}: {set(opt) - set(CO) - {'tag'}}"
    assert not (set(opt) - {"tag"}) or "tag" in opt, f"đổi cấu hình thì phải có tag: {luot}"
    assert int(sd) > 0
    assert rw in THUONG_CU + THUONG_MOI + THUONG_B2, f"thưởng lạ: {rw}"
    assert al in CU + CD + B1 + PP + B2 + ("sft",), f"thuật toán lạ: {al}"
    assert rw not in THUONG_B2 or al == "grpo", f"bcmr chỉ đi với --algo grpo (grpo_moi_b2 assert): {luot}"
    sai_b2 = [k for k in opt if k in CO_B2 and CO_B2[k] != al]
    assert not sai_b2, f"cờ {sai_b2} không thuộc {al}: {luot}"
    assert "lam" not in opt or al in PP, f"lam chỉ cho ppo/a2c: {luot}"
    assert not (set(opt) & CO_B1) or al in B1, f"cờ của grpo_moi_b1 chỉ cho {B1}: {luot}"
    if rw in THUONG_MOI:
        assert al in CU and set(opt) - {"tag"} <= CO_GL, f"thưởng mới chạy qua thuong_moi → grpo_luoi: chỉ {CU}, cờ {CO_GL}"
    return al, rw, sd, opt


def tep_cua(luot):
    al, rw, sd, opt = phan_tich(luot)
    if rw in THUONG_MOI:
        return "thuong_moi.py"
    if al in PP:
        return "grpo_ppo.py"
    if al == "sft":
        return "sft_tiep.py"
    if al in B1:
        return "grpo_moi_b1.py"
    if al in B2 or rw in THUONG_B2:
        return "grpo_moi_b2.py"
    # đúng luật của make_nb.tep_cua
    return "grpo_codien.py" if al in CD or set(opt) - {"tag"} - CO_GL or (al != "grpo" and rw != "spice") else "grpo_luoi.py"


def tep_phu_thuoc(luot):
    """Các tệp phải có trong thư mục làm việc để script chạy được."""
    t = tep_cua(luot)
    can = ["grpo_luoi.py", "grpo_spice.py", "build_branch_data.py"]
    can += {"grpo_codien.py": ["grpo_codien.py"], "grpo_ppo.py": ["grpo_codien.py", "grpo_ppo.py"],
            "sft_tiep.py": ["sft_tiep.py"], "grpo_moi_b1.py": ["grpo_codien.py", "grpo_moi_b1.py"],
            "grpo_moi_b2.py": ["grpo_codien.py", "grpo_moi_b2.py"],
            "thuong_moi.py": ["thuong_moi.py", "metric_exec.py", "descriptors_ten.jsonl"]}.get(t, [])
    return can


def ten_thu_muc(luot, probe=False):
    al, rw, sd, opt = phan_tich(luot)
    tag = opt.get("tag")
    return f"{'probe_' if probe else ''}{al}_{rw}_{sd}{'_' + tag if tag else ''}"


def tong_buoc(luot, probe=False):
    return 20 if probe else int(phan_tich(luot)[3].get("steps", 500))


def dung_lenh(luot, bundle, merged, out, probe=False, keys=None, py="python", bs=4, accum=4, gen_chunk=8):
    al, rw, sd, opt = phan_tich(luot)
    cmd = [py, tep_cua(luot), "--probe" if probe else "--train", "--no-q4", "--bs", str(bs), "--accum", str(accum),
           "--gen-chunk", str(gen_chunk), "--algo", al, "--reward", rw, "--seed", sd,
           "--bundle", bundle, "--merged", merged, "--out", out, "--resume", "auto"]
    for k, v in opt.items():
        if k != "tag":
            cmd += [CO[k], v]
    if sd == "101" and not probe and keys:
        cmd += ["--keys-ck500", keys]
    return cmd


def buoc_cua(p):
    return int(p.rstrip("/").rsplit("-", 1)[1])


def tep_them(luot):
    """Tệp bắt buộc thêm trong điểm lưu (ghi SAU ba tệp chính, thiếu thì chạy tiếp sẽ assert hoặc mất trạng thái).
    Mục bắt đầu bằng '-' là tệp chính KHÔNG có ở thuật toán đó (ES không có optimizer)."""
    al = phan_tich(luot)[0]
    return {"ppo": ("value_head.pt",), "a2c": ("value_head.pt",), "seqgan": ("discriminator.pt",),
            "quark": ("quark_pool.json",), "es": ("es_state.json", "-optimizer.pt")}.get(al, ())


def can_cua(them=()):
    bo = {f[1:] for f in them if f.startswith("-")}
    return tuple(f for f in TEP_CAN if f not in bo) + tuple(f for f in them if not f.startswith("-"))


def diem_luu_du(d, them=(), on_dinh=0):
    """Điểm lưu đủ TEP_CAN + `them`; on_dinh > 0: mọi tệp đã yên ≥ on_dinh giây (Trainer còn đang ghi thì bỏ qua)."""
    now = time.time()

    def yen(p):
        if not on_dinh:
            return True
        m = [os.path.getmtime(os.path.join(r, f)) for r, _, fs in os.walk(p) for f in fs]
        return bool(m) and now - max(m) >= on_dinh

    return sorted((p for p in glob.glob(os.path.join(d, "checkpoint-*"))
                   if os.path.isdir(p) and re.search(r"checkpoint-\\d+$", p)
                   and all(os.path.exists(os.path.join(p, f)) for f in can_cua(them)) and yen(p)), key=buoc_cua)


def dong_bo_len(out, dr, giu=3, log_ten="train.log", them=(), on_dinh=60):
    """Chép điểm lưu đủ mà Drive chưa có. Chép sang `.tam_checkpoint-N` rồi os.rename ⇒ trên Drive không bao giờ có
    điểm lưu dở (FUSE chỉ đẩy tệp đã đóng). Giữ `giu` điểm lưu cuối trên Drive. Trả danh sách bước vừa chép."""
    os.makedirs(dr, exist_ok=True)
    co = {buoc_cua(p) for p in diem_luu_du(dr, them)}
    moi = []
    moc = max(co) if co else -1
    for p in diem_luu_du(out, them, on_dinh):
        n = buoc_cua(p)
        if n <= moc:           # đã có, hoặc cũ hơn bản mới nhất trên Drive (đã bị tỉa theo `giu`): không chép lại
            continue
        tam = os.path.join(dr, f".tam_checkpoint-{n}")
        shutil.rmtree(tam, ignore_errors=True)
        shutil.copytree(p, tam)
        dich = os.path.join(dr, f"checkpoint-{n}")
        shutil.rmtree(dich, ignore_errors=True)      # bản dở do lần trước mất máy giữa lúc đổi tên
        os.rename(tam, dich)
        moi.append(n)
    du = diem_luu_du(dr, them)
    for p in du[:-giu] if giu else []:
        shutil.rmtree(p, ignore_errors=True)
    for f in (log_ten, "prompt_keys.json", "log_history.json"):
        s = os.path.join(out, f)
        if os.path.exists(s):
            shutil.copyfile(s, os.path.join(dr, f + ".tam"))
            os.replace(os.path.join(dr, f + ".tam"), os.path.join(dr, f + (".drive" if f == log_ten else "")))
    return moi


def khoi_phuc(dr, out, them=()):
    """Máy mới: chép điểm lưu đủ mới nhất từ Drive về `out` nếu local chưa có bản mới hơn. Trả bước, hoặc 0."""
    os.makedirs(out, exist_ok=True)
    du = diem_luu_du(dr, them)
    if not du:
        return 0
    n = buoc_cua(du[-1])
    co = [buoc_cua(p) for p in diem_luu_du(out, them)]
    if not co or max(co) < n:
        shutil.copytree(du[-1], os.path.join(out, f"checkpoint-{n}"), dirs_exist_ok=True)
    if os.path.exists(os.path.join(dr, "train.log.drive")) and not os.path.exists(os.path.join(out, "train.log")):
        shutil.copyfile(os.path.join(dr, "train.log.drive"), os.path.join(out, "train.log"))   # giữ lịch sử log
    return max(n, max(co) if co else 0)


SO = re.compile(r"^[-\\d.e+naif]+$")


def tien_do(log, tu=0):
    """Đọc train.log của lượt chạy NÀY: chỉ phần từ byte `tu` (cỡ log lúc bấm chạy) và sau dòng [tiếp từ] cuối."""
    L = []
    if os.path.exists(log):
        with open(log, encoding="utf-8", errors="ignore") as f:
            f.seek(tu)
            L = f.read().splitlines()
    i = max([k for k, l in enumerate(L) if l.startswith("[tiếp từ]")], default=-1)
    tiep = L[i] if i >= 0 else None
    goc = int(re.findall(r"checkpoint-(\\d+)", tiep)[0]) if tiep and "checkpoint-" in tiep else 0
    D = []
    for l in L[i + 1:] if i >= 0 else L:
        if l.startswith("{'loss'") or l.startswith("{'reward'"):
            try:
                D.append({k: float(v) for k, v in ast.literal_eval(l).items() if SO.match(str(v))})
            except Exception:
                pass
    nap = [l for l in L[i + 1:] if l.startswith("[nạp default]")] if i >= 0 else []
    m = re.search(r"→\\s*([\\d.]+)", nap[-1]) if nap else None
    xong = [l for l in L if l.startswith("[xong]")]
    x = D[-10:]
    st_b = [d["step_time"] for d in x if "step_time" in d]
    return dict(goc=goc, buoc=goc + len(D), n_log=len(D), tiep=tiep, resume=bool(tiep and "checkpoint-" in tiep),
                nap_default=(float(m.group(1)) if m else None), xong=xong[-1] if xong else None,
                s_buoc=(st.mean(st_b) if st_b else None),
                thuong=(st.mean(d["reward"] for d in x if "reward" in d) if any("reward" in d for d in x) else None),
                kl=(st.mean(d["kl"] for d in x if "kl" in d) if any("kl" in d for d in x) else None),
                loi=any(("Traceback" in l or "OutOfMemory" in l or "CUDA out of memory" in l or "Killed" in l) for l in L[-80:]))


def kiem_nap_default(td):
    """Lượt chạy tiếp phải có dòng [nạp default] với |lora_B| sau > 0 (lỗi 30/9). Trả (đạt, lý do)."""
    if td["tiep"] is None:
        return None, "chưa thấy dòng [tiếp từ] (đang nạp mô hình)"
    if not td["resume"]:
        return True, "lượt mới (tiếp từ None), không cần nạp default"
    if td["nap_default"] is None:
        return None, "chưa thấy dòng [nạp default]"
    return td["nap_default"] > 0, f"|lora_B| sau = {td['nap_default']}"


def doc_tham_do(log, tu=0):
    """Dòng [xong] của chế độ --probe: '… s/bước · đỉnh VRAM x GiB …' → (s_buoc, vram)."""
    td = tien_do(log, tu)
    if not td["xong"]:
        return None, None
    s = re.search(r"([\\d.]+) s/bước", td["xong"])
    v = re.search(r"đỉnh VRAM ([\\d.]+) GiB", td["xong"])
    return (float(s.group(1)) if s else None), (float(v.group(1)) if v else None)


def chon_may(gpu, vram, s_buoc, s_buoc_a100=None, s_buoc_l4=None, tran_l4=17.0, ti_so=1.5):
    """Luật CLAUDE.md (5/9, nới 9/9): CPU → T4 → L4 → A100; L4 khi đỉnh VRAM ≤ ~17 GiB và giờ_L4/giờ_A100 ≤ 1,5."""
    g = (gpu or "").upper()
    if "L4" in g:
        s_buoc_l4 = s_buoc
    if "A100" in g:
        s_buoc_a100 = s_buoc
    if vram is not None and vram > tran_l4 and "A100" not in g:
        return "A100", f"đỉnh VRAM {vram:.1f} GiB > {tran_l4} GiB (biên 25% trên 22 GiB của L4)"
    if s_buoc_l4 and s_buoc_a100:
        r = s_buoc_l4 / s_buoc_a100
        return ("L4" if r <= ti_so else "A100"), f"giờ_L4/giờ_A100 = {r:.2f} ({'≤' if r <= ti_so else '>'} {ti_so})"
    if "L4" in g:
        return "L4", "chưa đo A100; L4 đủ VRAM ⇒ giữ L4 (chỉ lên A100 khi đo được tỉ số > 1,5)"
    if "A100" in g:
        return "?", "chỉ có số A100; chạy thăm dò trên L4 rồi so (luật: thử card rẻ trước)"
    return "L4", f"{gpu}: chạy thăm dò lại trên L4 trước khi chọn"


def chay(cmd, log, cwd, env=None):
    """start_new_session=True: bấm Stop một ô notebook không gửi SIGINT sang tiến trình train (luật Colab ①)."""
    tu = os.path.getsize(log) if os.path.exists(log) else 0
    f = open(log, "a")
    p = subprocess.Popen(cmd, cwd=cwd, stdout=f, stderr=subprocess.STDOUT, env=env or os.environ.copy(),
                         start_new_session=True, text=True)
    p.tu_byte = tu      # theo_doi chỉ đọc phần log của lần chạy này
    return p


def gio_vn(t=None):
    return time.strftime("%d/%m %H:%M", time.gmtime((t or time.time()) + 7 * 3600))


def dung(p, cho=120):
    """Dừng nhẹ (SIGTERM cả nhóm tiến trình) để Trainer kịp thoát; quá `cho` giây thì SIGKILL."""
    if p.poll() is not None:
        return p.returncode
    try:
        os.killpg(os.getpgid(p.pid), signal.SIGTERM)
    except ProcessLookupError:
        pass
    try:
        p.wait(cho)
    except subprocess.TimeoutExpired:
        os.killpg(os.getpgid(p.pid), signal.SIGKILL)
        p.wait()
    return p.returncode


def theo_doi(p, out, dr, tong, t0, cat_gio=None, in_moi=120, dong_bo_moi=300, log_ten="train.log", inra=print,
             them=(), on_dinh=60):
    """Ô foreground suốt lượt (luật Colab ④). Nhật ký cuộn có mốc giờ VN, không clear_output, flush.
    Trả (mã thoát, td cuối, kiểm nạp default)."""
    log = os.path.join(out, log_ten)
    tu = getattr(p, "tu_byte", 0)
    t_db = time.time()
    da_kiem = False
    kq_nap = (True, "")
    while p.poll() is None:
        time.sleep(in_moi)
        td = tien_do(log, tu)
        con = (tong - td["buoc"]) * td["s_buoc"] / 3600 if td["s_buoc"] else None
        inra(f"{gio_vn()} VN · {(time.time() - t0) / 3600:5.2f} h · bước {td['buoc']}/{tong} (tiếp từ {td['goc']})"
             f" · {td['s_buoc'] or 0:.0f} s/bước · còn {'?' if con is None else f'{con:.1f} h'}"
             f" · thưởng {td['thuong'] if td['thuong'] is not None else float('nan'):.3f}"
             f" · kl {td['kl'] if td['kl'] is not None else float('nan'):.4f}"
             f" · Drive: {', '.join(os.path.basename(x) for x in diem_luu_du(dr, them)[-1:]) or 'chưa có'}", flush=True)
        if not da_kiem:
            kq_nap = kiem_nap_default(td)
            if kq_nap[0] is not None:
                da_kiem = True
                inra(f"   [kiểm nạp default] {'ĐẠT' if kq_nap[0] else '⛔ KHÔNG ĐẠT'} · {kq_nap[1]}", flush=True)
                if not kq_nap[0]:
                    inra("   ⛔ policy về lại S1 (lỗi 30/9) ⇒ dừng lượt", flush=True)
                    dung(p)
                    break
        if td["loi"]:
            inra("   ⚠️ Traceback/OOM ở cuối log", flush=True)
        if time.time() - t_db >= dong_bo_moi:
            moi = dong_bo_len(out, dr, log_ten=log_ten, them=them, on_dinh=on_dinh)
            if moi:
                inra(f"   [Drive] đã chép checkpoint-{moi}", flush=True)
            t_db = time.time()
        if cat_gio and time.time() - t0 > cat_gio * 3600:
            inra("   ⛔ chạm trần giờ đặt trước, dừng nhẹ", flush=True)
            dung(p)
    moi = dong_bo_len(out, dr, log_ten=log_ten, them=them, on_dinh=0)   # tiến trình đã thoát: không còn ai ghi
    td = tien_do(log, tu)
    inra(f"{gio_vn()} VN · KẾT THÚC · mã thoát {p.returncode} · bước {td['buoc']}/{tong} · Drive chép thêm {moi}", flush=True)
    return p.returncode, td, kq_nap


def nen(out, dich, giu_ck=1):
    """Zip thư mục kết quả (luật: ô cuối zip riêng thư mục kết quả). Giữ `giu_ck` điểm lưu cuối kèm optimizer
    (đủ để chạy tiếp), các điểm lưu cũ chỉ giữ adapter + trainer_state. Ghi sang tên tạm rồi đổi tên."""
    import zipfile
    du = diem_luu_du(out)
    giu = {os.path.basename(p) for p in du[-giu_ck:]} if giu_ck else set()
    tam = dich + ".tam"
    n = 0
    with zipfile.ZipFile(tam, "w", zipfile.ZIP_DEFLATED) as z:
        for r, _, fs in os.walk(out):
            for f in fs:
                p = os.path.join(r, f)
                rel = os.path.relpath(p, os.path.dirname(out))
                ck = next((x for x in rel.split(os.sep) if x.startswith("checkpoint-")), None)
                if ck and ck not in giu and f in ("optimizer.pt", "scheduler.pt", "rng_state.pth"):
                    continue
                z.write(p, rel)
                n += 1
    os.replace(tam, dich)
    return n


class TienTrinhCu:
    """Bọc một PID còn sống từ lần chạy ô trước (cùng máy ảo) để theo_doi dùng như Popen."""
    def __init__(self, pid, log):
        self.pid, self.returncode = pid, None
        self.tu_byte = 0
        self.log = log

    def poll(self):
        try:
            os.kill(self.pid, 0)
            return None
        except ProcessLookupError:
            self.returncode = self.returncode if self.returncode is not None else -999   # không biết mã thật
            return self.returncode

    def wait(self, timeout=None):
        t = time.time()
        while self.poll() is None:
            if timeout and time.time() - t > timeout:
                raise subprocess.TimeoutExpired("pid", timeout)
            time.sleep(2)
        return self.returncode


def tim_tien_trinh(out):
    """PID của tiến trình train đang ghi vào `out` (bấm Run all lại khi lượt cũ còn sống thì KHÔNG chạy bản thứ hai)."""
    r = subprocess.run(["pgrep", "-f", "--", f"--out {out} "], capture_output=True, text=True)
    me = os.getpid()
    return [int(x) for x in r.stdout.split() if int(x) != me]
''')
sys.path.insert(0, W)
import importlib, colab_logic as C
importlib.reload(C)
assert CHE_DO in ("tham_do", "train"), CHE_DO
PROBE = CHE_DO == "tham_do"
al, rw, sd, opt = C.phan_tich(LUOT)
TEN = C.ten_thu_muc(LUOT, probe=PROBE)
TONG = C.tong_buoc(LUOT, probe=PROBE)
CAN = C.tep_phu_thuoc(LUOT)
print(f"lượt {LUOT} · script {C.tep_cua(LUOT)} · thư mục {TEN} · {TONG} bước · cần {CAN}", flush=True)
if al in ("gspo",) and "steps" not in opt:
    print("⚠️ gspo ép μ=2: muốn đi qua đủ 1.000 câu nhắc như ck500 thì đặt steps=1000", flush=True)


In [ ]:
# Ô 3. Môi trường (khoảng 5 phút)
import subprocess
from google.colab import drive
drive.mount("/content/drive")
os.makedirs(DRIVE_GOC, exist_ok=True)
def sh(c, duoi=1500):
    r = subprocess.run(c, shell=True, capture_output=True, text=True)
    print((r.stdout + r.stderr)[-duoi:], flush=True); return r.returncode
GPU = subprocess.run("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader", shell=True, capture_output=True, text=True).stdout.strip()
print("GPU:", GPU, flush=True); assert GPU, "DUNG: chua bat GPU"
if "T4" in GPU:
    print("⚠️ T4 chạy fp16 như Kaggle; L4/A100 chạy bf16 nên số có thể lệch nhẹ so với lượt Kaggle cùng tên, phải khai khi báo", flush=True)
sh("apt-get -qq update && apt-get -qq install -y openjdk-8-jre-headless")
sh("update-alternatives --set java /usr/lib/jvm/java-8-openjdk-amd64/jre/bin/java")
assert "1.8" in subprocess.run("java -version", shell=True, capture_output=True, text=True).stderr, "DUNG: SPICE can Java 8"
sh(f"{sys.executable} -m pip install -q 'transformers==5.20.2' 'peft==0.21.2' 'trl==0.29.1' accelerate bitsandbytes 'kaggle==2.2.4' "
   f"pycocoevalcap sacrebleu scipy{' bert_score' if rw == 'bertscore' else ''} 2>&1 | tail -3")
sh(f"{sys.executable} -m pip uninstall -y torchao 2>&1 | tail -1")
assert sh(f"""{sys.executable} -c "import trl, transformers, peft, torch; print(trl.__version__, transformers.__version__, peft.__version__, torch.__version__, torch.cuda.get_device_name(0)); assert (trl.__version__, transformers.__version__, peft.__version__) == ('0.29.1', '5.20.2', '0.21.2'), 'lech phien ban'" """) == 0, "DUNG: moi truong loi"
print("✅ môi trường", flush=True)


In [ ]:
# Ô 4. Dữ liệu + kiểm md5 + tự kiểm + hoà S1 (khoảng 15 phút; hoà xong thì lần sau bỏ qua nếu máy chưa mất)
import getpass, stat, glob, shutil, hashlib, json, importlib.util
KD = "/root/.kaggle"; os.makedirs(KD, exist_ok=True); TKF = f"{KD}/access_token"
if not os.path.exists(TKF):
    tok = None
    try:
        from google.colab import userdata
        tok = userdata.get("KAGGLE_TOKEN")
    except Exception:
        pass
    tok = tok or getpass.getpass(f"Khoa Kaggle cua {TK} (KGAT_...): ")
    with open(TKF, "w") as f: f.write(tok.strip())
    os.chmod(TKF, stat.S_IRUSR | stat.S_IWUSR); del tok
ENV_K = {**os.environ, "KAGGLE_CONFIG_DIR": KD, "KAGGLE_API_TOKEN": TKF}
def tai(ds, dich):
    if os.path.exists(dich + "/.xong"): print("đã có", ds, flush=True); return
    os.makedirs(dich, exist_ok=True); t = time.time()
    r = subprocess.run(["kaggle", "datasets", "download", ds, "-p", dich, "--unzip"], env=ENV_K, capture_output=True, text=True)
    print((r.stdout + r.stderr)[-300:], flush=True); assert r.returncode == 0, f"DUNG: khong tai duoc {ds}"
    for z in glob.glob(dich + "/*.zip"):
        d = z[:-4]; os.makedirs(d, exist_ok=True); shutil.unpack_archive(z, d); os.remove(z)
    open(dich + "/.xong", "w").close(); print(f"tải {ds} · {(time.time()-t)/60:.1f} phút", flush=True)
D = "/content/ds"
tai(f"{TK}/grpo-luoi-script", f"{D}/script")
tai(f"{TK}/luoi-kit", f"{D}/kit")
tai(f"{TK}/fgrb-p1-bundle", f"{D}/bundle")                  # khoảng 2,7 GB
md5 = lambda p: hashlib.md5(open(p, "rb").read()).hexdigest()
BUNDLE = next(r for r, d, f in os.walk(f"{D}/bundle") if "adapter_s1_seed101" in d and "images" in d)
assert md5(f"{BUNDLE}/adapter_s1_seed101/adapter_model.safetensors") == "0c60a4418d54eaf27bf1d27b895538b7", "DUNG: adapter S1 lech"
MD5 = {'grpo_luoi.py': '92d0a9bc6b8cae8a480aef08f99fb9d8', 'grpo_spice.py': '8489108147c8d1ed80c3dc67c8ad3d3f', 'build_branch_data.py': '619e63e123a6dbf60086e65ee94a3912', 'prompt_keys_ck500.json': '4bf30d816dc531842495535828dcbb43', 'grpo_codien.py': '5499d6041bf0fb7c780099abcaa27f32', 'grpo_ppo.py': 'e3330d4469f6b60c9e3795e2d46b83db', 'sft_tiep.py': '500da31dbe8d46505abc717be956926f', 'grpo_moi_b1.py': 'f0f83534642620c738353bbc3d86c27e', 'grpo_moi_b2.py': '821411b88e1f18fc8d1f3062e5e21f06', 'thuong_moi.py': 'dc88473e9115f8bcc954fb402f43a368', 'metric_exec.py': '9bf0b84145458fd55919a5e161b9766f', 'descriptors_ten.jsonl': 'a3535c64534cc88f923cb36da102d91e'}
def nguon(f):
    k = glob.glob(f"{D}/script/**/{f}", recursive=True) + glob.glob(f"{D}/kit/**/{f}", recursive=True)
    assert len(k) >= 1, f"DUNG: thieu {f}"
    return k[0]
for f in CAN:
    shutil.copy(nguon(f), f"{W}/{f}")
    assert md5(f"{W}/{f}") == MD5[f], f"DUNG: {f} lech ban da kiem (kit/script doi thi dung lai notebook bang make_colab_nb.py)"
KEYS = nguon("prompt_keys_ck500.json"); assert md5(KEYS) == MD5["prompt_keys_ck500.json"]
SPL = list(importlib.util.find_spec("pycocoevalcap").submodule_search_locations)[0] + "/spice/lib"
for j, sz in {"stanford-corenlp-3.6.0.jar": 7270741, "stanford-corenlp-3.6.0-models.jar": 378116526}.items():
    if not (os.path.exists(f"{SPL}/{j}") and os.path.getsize(f"{SPL}/{j}") == sz):
        shutil.copy(nguon(j), f"{SPL}/{j}")
    assert os.path.getsize(f"{SPL}/{j}") == sz, f"DUNG: {j} sai kich thuoc"
def py(*a):
    r = subprocess.run([sys.executable, *a], cwd=W, capture_output=True, text=True)
    print((r.stdout + r.stderr)[-1200:], flush=True); return r.returncode
assert py("grpo_luoi.py", "--selftest-algo") == 0, "DUNG: selftest algo"
for f, co in (("grpo_codien.py", "--selftest-algo"), ("grpo_ppo.py", "--selftest-algo"), ("grpo_moi_b1.py", "--selftest-algo"),
              ("grpo_moi_b2.py", "--selftest-algo")):
    if f in CAN:
        assert py(f, co) == 0, f"DUNG: selftest {f}"
if "thuong_moi.py" in CAN:
    assert py("thuong_moi.py", "--selftest-moi", "--bundle", W) == 0, "DUNG: selftest thuong moi"
else:
    assert py("grpo_moi_b2.py" if rw == "bcmr" else "grpo_luoi.py", "--selftest-thuong", "--bundle", BUNDLE, "--rewards", rw) == 0, "DUNG: selftest thuong"
MERGED = "/content/s1_merged"
if not os.path.exists(f"{MERGED}/config.json"):
    assert py("grpo_spice.py", "--merge", "--bundle", BUNDLE, "--merged", MERGED) == 0, "DUNG: hoa loi"
print(f"✅ dữ liệu · BUNDLE {BUNDLE} · {(time.time()-T0)/60:.0f} phút từ đầu", flush=True)


In [ ]:
# Ô 5. Thăm dò (chỉ khi CHE_DO = "tham_do"): 20 bước trên 50 câu nhắc dài nhất ⇒ s/bước, đỉnh VRAM, chọn máy
THEM = C.tep_them(LUOT)          # tệp riêng của điểm lưu: value_head.pt (PPO/A2C) · discriminator.pt · quark_pool.json · es_state.json (ES không có optimizer)
ENV = {**os.environ, "CUDA_VISIBLE_DEVICES": "0", "TQDM_DISABLE": "1", "HF_HUB_DISABLE_PROGRESS_BARS": "1", "PYTHONUNBUFFERED": "1",
       "PYTORCH_ALLOC_CONF": "expandable_segments:True", "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True"}
TD_JSON = f"{DRIVE_GOC}/tham_do_{C.ten_thu_muc(LUOT)}.json"
if PROBE:
    OUT = f"{W}/{TEN}"; shutil.rmtree(OUT, ignore_errors=True); os.makedirs(OUT)
    p = C.chay(C.dung_lenh(LUOT, BUNDLE, MERGED, OUT, probe=True, py=sys.executable), f"{OUT}/train.log", W, ENV)
    rc, td, _ = C.theo_doi(p, OUT, f"{DRIVE_GOC}/{TEN}", TONG, T0, in_moi=60, dong_bo_moi=300, them=THEM)
    s, v = C.doc_tham_do(f"{OUT}/train.log", p.tu_byte)
    if rc != 0 or s is None:
        print(open(f"{OUT}/train.log", errors="ignore").read()[-3000:], flush=True)
        raise SystemExit("DUNG: tham do loi (OOM thi doc dinh VRAM o tren; L4 tran thi chay lai tren A100)")
    D_TD = json.load(open(TD_JSON)) if os.path.exists(TD_JSON) else {}
    D_TD[GPU.split(",")[0]] = dict(s_buoc=s, vram=v, luc=C.gio_vn())
    json.dump(D_TD, open(TD_JSON + ".tam", "w"), ensure_ascii=False, indent=1); os.replace(TD_JSON + ".tam", TD_JSON)
    sa = next((x["s_buoc"] for g, x in D_TD.items() if "A100" in g), None)
    sl = next((x["s_buoc"] for g, x in D_TD.items() if "L4" in g), None)
    may, ly_do = C.chon_may(GPU, v, s, s_buoc_a100=sa, s_buoc_l4=sl)
    tong = C.tong_buoc(LUOT)
    print(f"\n===== THĂM DÒ {LUOT} trên {GPU}: {s:.1f} s/bước · đỉnh VRAM {v:.2f} GiB", flush=True)
    print(f"   {tong} bước ≈ {tong * s / 3600:.1f} h máy này [suy: s/bước đo trên câu nhắc DÀI NHẤT nên là cận trên]", flush=True)
    print(f"   các máy đã đo: {D_TD}", flush=True)
    print(f"   ⇒ khuyến nghị: {may} · {ly_do}", flush=True)
    print("   Đơn vị Colab = giờ × đơn vị/giờ đang hiện ở Colab lúc chạy (giá phải đọc tại chỗ, đừng nhớ).", flush=True)
else:
    print("bỏ qua thăm dò (CHE_DO = train) · đã đo:", json.load(open(TD_JSON)) if os.path.exists(TD_JSON) else "chưa có", flush=True)


In [ ]:
# Ô 6. Train (chỉ khi CHE_DO = "train") — FOREGROUND suốt lượt, đừng bấm Stop, giữ tab mở
if not PROBE:
    OUT, DR = f"{W}/{TEN}", f"{DRIVE_GOC}/{TEN}"
    os.makedirs(DR, exist_ok=True)
    if os.path.exists(f"{DR}/final/adapter_model.safetensors"):
        print(f"✅ lượt đã xong từ trước: {DR}/final — không chạy lại", flush=True)
    else:
        cu = C.tim_tien_trinh(OUT)
        if cu:
            print(f"⚠️ tiến trình train cũ còn sống (pid {cu}) ⇒ chỉ theo dõi tiếp, KHÔNG chạy bản thứ hai", flush=True)
            p = C.TienTrinhCu(cu[0], f"{OUT}/train.log")
        else:
            n = C.khoi_phuc(DR, OUT, THEM)
            print(f"khôi phục từ Drive: {'checkpoint-' + str(n) if n else 'không có, chạy từ đầu'} · Drive có {[os.path.basename(x) for x in C.diem_luu_du(DR, THEM)]}", flush=True)
            cmd = C.dung_lenh(LUOT, BUNDLE, MERGED, OUT, keys=KEYS, py=sys.executable)
            print("lệnh:", " ".join(cmd), flush=True)
            p = C.chay(cmd, f"{OUT}/train.log", W, ENV)
        rc, td, nap = C.theo_doi(p, OUT, DR, TONG, T0, cat_gio=CAT_GIO, in_moi=120, dong_bo_moi=300, them=THEM)
        print("\n".join(l for l in open(f"{OUT}/train.log", errors="ignore").read().splitlines()[-25:]
                        if not l.startswith(("PTB", "Parsing", "Threads", "Loading"))), flush=True)
        if os.path.exists(f"{OUT}/final/adapter_model.safetensors"):
            shutil.rmtree(f"{DR}/.tam_final", ignore_errors=True)
            shutil.copytree(f"{OUT}/final", f"{DR}/.tam_final"); shutil.rmtree(f"{DR}/final", ignore_errors=True)
            os.rename(f"{DR}/.tam_final", f"{DR}/final")
            print(f"✅ XONG {LUOT} · {td['buoc']}/{TONG} bước · final đã lên Drive", flush=True)
        else:
            print(f"⏸ chưa xong ({td['buoc']}/{TONG}, mã {rc}, kiểm nạp {nap}) · điểm lưu cuối trên Drive: "
                  f"{[os.path.basename(x) for x in C.diem_luu_du(DR, THEM)][-1:]} · Run all lại để chạy tiếp", flush=True)


In [ ]:
# Ô 7. Zip kết quả lên Drive (giữ optimizer của điểm lưu cuối để còn chạy tiếp)
OUT, DR = f"{W}/{TEN}", f"{DRIVE_GOC}/{TEN}"
if os.path.isdir(OUT):
    z = f"/content/{TEN}_ket_qua.zip"
    n = C.nen(OUT, z, giu_ck=1)
    os.makedirs(DR, exist_ok=True)
    shutil.copyfile(z, f"{DR}/{TEN}_ket_qua.zip.tam"); os.replace(f"{DR}/{TEN}_ket_qua.zip.tam", f"{DR}/{TEN}_ket_qua.zip")
    print(f"✅ {n} tệp · {os.path.getsize(z)/2**20:.0f} MB → {DR}/{TEN}_ket_qua.zip", flush=True)
